# GSB 5544 — Topic 10.2: Regularization — Ridge, LASSO, Tuning Parameters and Cross-Validation — SOLUTION
*A 15–20 minute think-and-discuss opener before the penalized-regression walkthrough and PA 10.2. One running
example, thirteen short questions, no derivations.*

## How to use this

**The running example.** A regional coffee chain has 100 locations. For each store we know **ten things** about
the store and its neighborhood — foot traffic, ad spend, store size, seats, parking spaces, neighborhood income,
competitor count, social-media posts, average temperature, manager tenure — and **last week's sales**. The chain
is opening new stores and wants a **sales forecast from a store's characteristics alone**. 60 stores are the
training data; 40 are locked away as a test set. (This is the example on the interactive page
[*Shrink to Fit*](https://gato365.github.io/gsb5544_instructor_learn_prep/week_10/shrink_to_fit.html), where the sliders show every number in this notebook moving; the questions below are
also in the page's *Think · Discuss · Reveal* box.)

**The rhythm.** For each question: think for a minute, discuss with a neighbour for two, then the instructor
reveals the answer. Thirteen questions, roughly 90 seconds each.

**The three objectives**, so they are in front of us the whole time. Each adds a term to the one before it.

$$\text{Least squares:}\quad \min_{\beta_0,\boldsymbol\beta} \underbrace{\sum_{i=1}^{n} (y_i - \hat y_i)^2}_{\text{prediction error on the training rows}}$$

$$\text{Ridge:}\quad \min_{\beta_0,\boldsymbol\beta} \Big[\underbrace{\sum_{i=1}^{n} (y_i - \hat y_i)^2}_{\text{prediction error}} + \underbrace{\lambda \sum_{j=1}^{p} \beta_j^2}_{\text{penalty: squared coefficients}}\Big]$$

$$\text{LASSO:}\quad \min_{\beta_0,\boldsymbol\beta} \Big[\underbrace{\sum_{i=1}^{n} (y_i - \hat y_i)^2}_{\text{prediction error}} + \underbrace{\lambda \sum_{j=1}^{p} |\beta_j|}_{\text{penalty: absolute coefficients}}\Big]$$

| symbol | what it is |
|---|---|
| $y_i$, $\hat y_i$ | actual and predicted weekly sales for training store $i$; $n = 60$ |
| $\beta_1, \dots, \beta_p$ | the **coefficients**, one per predictor ($p = 10$); learned by fitting |
| $\beta_0$ | the intercept — **not penalized**; it only sets the overall level of sales |
| $\lambda \ge 0$ | the **tuning parameter**: how heavily coefficient size is charged. Not learned by fitting; chosen by cross-validation |

> **About `alpha`.** scikit-learn calls $\lambda$ `alpha`, and it scales the error term differently for different
> models (`Lasso` and `ElasticNet` divide it by $2n$; `Ridge` does not). So `alpha=1` is **not** the same strength of
> penalty for ridge and for LASSO, and a good `alpha` on one data set says nothing about another. Compare penalty
> strengths by their cross-validation curves, never by the number itself.

---
**Q1.** Plain linear regression. We fit a line (well, a plane) to the 60 training stores: weekly sales ≈ intercept + coefficients × the ten store characteristics. What exactly does least squares minimize, and why might a model that fits those 60 stores very well still forecast a *new* store poorly?

**Answer:** Least squares picks the intercept and coefficients that make the **sum of squared training errors** as small as possible. It is judged only on the 60 stores it saw, so it will happily bend to whatever is peculiar about them — a few unusual stores, a lucky correlation — and those peculiarities do not repeat in the next store. Low training error is a fact about the past sample; forecasting is a claim about new stores.

*Follow-up:* Which number did we trust last week instead of training error? (Cross-validated test error.)

---
**Q2.** Overfitting through the coefficients. Three of our ten predictors — seats, parking spaces and store size — all describe how big the store is, and two others may be pure noise. How can coefficients become overly sensitive to the particular 60 stores we happened to sample?

**Answer:** When predictors overlap, many different combinations of coefficients give almost the same fitted values: a large positive coefficient on seats and a large negative one on store size can cancel. Least squares picks whichever combination fits *this* sample best, so a different 60 stores can flip the signs and sizes completely. Noise predictors get non-zero coefficients for the same reason — there is always some chance pattern to fit. Many predictors relative to rows makes all of this worse.

---
**Q3.** Why accept a worse fit? Suppose we deliberately pull the coefficients back toward zero, so training error goes *up* a little. Why could that make the forecasts for new stores *better*?

**Answer:** Part of what the unconstrained coefficients capture is signal and part is sample noise. Pulling them toward zero gives up a little of the signal (a bit more training error) but throws away a lot of the noise, so the model's predictions vary less from sample to sample. For new stores the trade is usually favourable — up to a point. Last week's picture applies: training error can only get worse as we restrain the model, while validation error falls first and rises later.

*Follow-up:* Does it *guarantee* better forecasts? No — it is a bet that the noise outweighs the lost signal, and cross-validation is how we check the bet.

---
**Q4.** The penalty. Regularization turns that idea into a single thing to minimize: training error **plus** a penalty on coefficient size. In words, what are the two terms fighting over, and what decides who wins?

**Answer:** The first term rewards fitting the 60 stores closely; the second charges a price for large coefficients. Minimizing their sum finds coefficients that fit well *for their size*. The exchange rate between the two is the tuning parameter $\lambda$ (scikit-learn calls it `alpha`): a larger $\lambda$ makes coefficient size more expensive, so the solution shrinks. The intercept is left out of the penalty — it just sets the overall level of sales and shrinking it would bias every forecast.

---
**Q5.** Ridge regression penalizes the **sum of squared coefficients**, $\lambda \sum_j \beta_j^2$. Why does that pull every coefficient toward zero, and why does it pull the biggest ones hardest?

**Answer:** Squaring makes a coefficient of 4 cost sixteen times a coefficient of 1, so the cheapest way to lower the penalty is to trim the largest coefficients first. Every coefficient is shrunk somewhat, and the trimming of a coefficient that is already small buys almost nothing, so ridge rarely drives anything exactly to zero — it approaches zero smoothly. With overlapping predictors like seats, parking and size, ridge tends to spread the shared effect across the group instead of letting one coefficient balloon and another go negative.

---
**Q6.** LASSO penalizes the **sum of absolute coefficients**, $\lambda \sum_j |\beta_j|$. How is the effect different, and why can LASSO set a coefficient to *exactly* zero when ridge does not?

**Answer:** With absolute values, moving a coefficient from 0.1 to 0 saves the same amount of penalty as moving it from 4.1 to 4 — the last step to zero is never cheaper than the others. If a predictor is not earning its keep in the fit, the penalty wins and the coefficient lands on zero and stays there. (Under ridge that last step from 0.1 to 0 saves only 0.01, so it is almost never worth taking.) Predictors with a zero coefficient are out of the model, which is why LASSO is described as doing variable selection.

---
**Q7.** Compare them. What do ridge and LASSO share, and what is the practical difference between *shrinking* a coefficient and *zeroing* it?

**Answer:** Both minimize training error plus a $\lambda$-weighted penalty on coefficient size, both need standardized predictors, both have $\lambda$ chosen by cross-validation, and neither penalizes the intercept. A shrunk coefficient keeps the predictor in the forecast with a muted effect; a zeroed one removes it, which gives a smaller model that is easier to explain and cheaper to feed with data. Among overlapping predictors LASSO often keeps one and drops the rest somewhat arbitrarily; ridge keeps them all with smaller coefficients. Which forecasts better depends on the data and is settled by cross-validation, not by preference.

*Follow-up:* A LASSO zero on `store size` while `seats` is kept does **not** mean store size is irrelevant or has no effect — LASSO chose a stand-in. Regularization is a forecasting tool, not a hypothesis test. (A mixture of the two penalties, elastic net, exists for exactly this correlated-group situation; the walkthrough shows it.)

---
**Q8.** Units. Store size is measured in square feet (hundreds to thousands); manager tenure in years (0 to 20). Why does that matter once we start penalizing coefficient size, and what do we do about it?

**Answer:** A coefficient's size depends on its predictor's units: sales per square foot is a tiny number, sales per year of tenure a large one, and neither says which predictor matters more. A penalty that treats every coefficient alike would therefore punish tenure heavily and square footage hardly at all. **Standardizing** each predictor (subtract its mean, divide by its standard deviation) puts every coefficient in the same units — dollars of sales per one standard deviation — so the penalty is fair and the coefficients are comparable. Large coefficients are not automatically a problem; they are only meaningful after this step.

*Follow-up:* Inside cross-validation the scaler must be fitted on each training fold alone (a pipeline does this); scaling all the rows first leaks the validation rows into the fit.

---
**Q9.** The tuning parameter. What does the model become when $\lambda = 0$, when $\lambda$ is small, and when $\lambda$ is enormous? And how is $\lambda$ different in kind from the coefficients?

**Answer:** At $\lambda = 0$ the penalty vanishes and we are back to ordinary least squares. For small $\lambda$ the coefficients are gently shrunk and predictions barely change. As $\lambda$ grows the coefficients head to zero (LASSO drops predictors one by one) and the forecast collapses toward the average sales of the training stores — a model that ignores the store. The coefficients are **learned** by fitting, given a $\lambda$; $\lambda$ itself is not learned from the fit — it is a **tuning parameter**, set from outside and chosen by comparing candidate values on validation data.

*Follow-up:* Software scales the error term differently (scikit-learn's `Lasso` divides it by $2n$, `Ridge` does not), so `alpha = 1` is not the same strength of penalty for ridge and LASSO, nor the same across data sets. The value is meaningless on its own; the cross-validation curve is what matters.

---
**Q10.** Choosing $\lambda$ by cross-validation. Here are six candidate penalty strengths and the 5-fold cross-validation MSE each produced on the 60 training stores (hypothetical numbers, in millions of dollars²):

| $\lambda$ | 0 | 0.1 | 1 | 10 | 100 | 1000 |
|---|---|---|---|---|---|---|
| CV MSE | 9.8 | 9.1 | 8.4 | 8.2 | 9.6 | 15.0 |

Which would you pick, and why? Why could training MSE not have made this choice?

**Answer:** Pick $\lambda = 10$: it has the lowest cross-validated error, and the curve has the expected shape — unpenalized at 9.8, improving as the penalty bites, then collapsing (15.0) when the penalty crushes the useful coefficients too. The strongest penalty is clearly not the best. Training MSE would have chosen $\lambda = 0$ every time, because any penalty can only make the fit to the training stores worse. One caution: 8.4 and 8.2 are close, and five folds give a noisy average, so $\lambda = 1$ would also be defensible; some analysts take the most penalized value whose error is within the noise of the minimum.

---
**Q11.** Grid search. We just compared six values of $\lambda$ by hand. Describe what `GridSearchCV` does with that list, in one or two sentences.

**Answer:** It takes the list of candidate values (the grid), and for each one runs the *same* K-fold cross-validation — same folds for every candidate — records the mean validation score, picks the candidate with the best mean, and refits that model on all the training rows. It is the table above, automated, with the bookkeeping (`cv_results_`, `best_params_`, `best_estimator_`) kept for you. Two tuning parameters (as in elastic net) just make the grid two-dimensional.

---
**Q12.** Validation versus test. The chain locked 40 stores away before we started. If cross-validation already tells us how well each $\lambda$ forecasts unseen stores, why keep those 40 untouched until the very end?

**Answer:** Because we *used* the validation scores to choose. The best of six candidates is partly the luckiest of six, so its cross-validated error is a little optimistic — and if we also chose features, methods and $\lambda$ on it, more so. The 40 held-out stores took no part in any decision, so the one error we compute on them after refitting the chosen model is an honest estimate of performance on genuinely new stores. Open the test set once, report the number, and do not go back and re-tune.

---
**Q13.** Put it together. In four or five sentences, explain to a manager how regularization, ridge/LASSO, the tuning parameter and cross-validation work together to produce a sales forecast for a store the chain has never seen.

**Answer:** We fit a linear model to the training stores but charge a price for large coefficients, which stops the model from memorizing quirks of our sample (regularization). Ridge charges by squared size and shrinks everything a little; LASSO charges by absolute size and can drop predictors entirely, giving a simpler model. How heavy the charge should be — the tuning parameter — is not something the fit can tell us, so we try a grid of values and let cross-validation on the training stores vote for the one that forecasts held-out stores best. We refit with that value on all the training stores, check it once on the stores we locked away, and that model, with standardized inputs, is what forecasts the new store.

---
## Misconceptions to name out loud

| students often think… | but… |
|---|---|
| "A big coefficient means that predictor is important / is a problem." | Coefficient size depends on the predictor's units until the predictors are standardized. Size is only comparable after that, and even then size is about the fit, not about importance in any causal sense. |
| "Regularization makes the model better." | It trades a little fit for less variance. Usually that helps on new data; sometimes (clean data, few predictors, many rows) it does nothing or hurts. Cross-validation tells you which. |
| "More penalty is safer." | Too much penalty flattens the model toward "predict the average sales": the CV curve rises again. The best penalty is the one with the lowest validation error, often moderate. |
| "Ridge and LASSO both do variable selection." | Ridge shrinks toward zero but essentially never reaches it; LASSO can land exactly on zero. Only LASSO (and elastic net) removes predictors. |
| "LASSO set `store size` to zero, so store size doesn't matter." | It means store size was not needed *for prediction given the other predictors* — very likely because `seats` carries the same information. It is not evidence of no effect and certainly not of no causal effect. |
| "Standardize once, then cross-validate." | The scaler is fitted; fit it on all rows and every validation fold has leaked its mean and spread into training. Put the scaler in the pipeline so it is refitted inside each training fold. |

---
## One tiny demonstration (optional, two minutes)

A simulated coffee chain in eight lines: five real predictors, one correlated copy (`seats` tracks `size`), one
pure-noise column. Standardize, then fit least squares, ridge and LASSO at one penalty and compare the
coefficients. Watch which method shrinks and which zeros. (The `alpha` values differ between the two models on
purpose — see the note above.)

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(5544)
n = 60
stores = pd.DataFrame({
    "traffic": rng.normal(800, 200, n),  "ads": rng.normal(300, 100, n),
    "size":    rng.normal(1800, 400, n), "income": rng.normal(65, 12, n),
    "competitors": rng.integers(0, 6, n),
})
stores["seats"]  = stores["size"] / 30 + rng.normal(0, 4, n)      # a correlated copy of size
stores["tenure"] = rng.normal(5, 3, n)                               # pure noise
sales = (6 * stores["traffic"] + 5 * stores["ads"] + 2 * stores["size"] + 60 * stores["income"]
         - 400 * stores["competitors"] + rng.normal(0, 2800, n))

fits = {
    "least squares": make_pipeline(StandardScaler(), LinearRegression()),
    "ridge":         make_pipeline(StandardScaler(), Ridge(alpha=10)),
    "LASSO":         make_pipeline(StandardScaler(), Lasso(alpha=300)),
}
coefs = pd.DataFrame({name: pipe.fit(stores, sales)[-1].coef_ for name, pipe in fits.items()}, index=stores.columns)
coefs.round(0) + 0.0        # + 0.0 turns LASSO's "-0.0" into a plain 0

,least squares,ridge,LASSO
traffic,1667.0,1418.0,1289.0
ads,449.0,342.0,87.0
size,347.0,219.0,0.0
income,1401.0,1200.0,1042.0
competitors,-753.0,-637.0,-409.0
seats,234.0,223.0,106.0
tenure,-141.0,-157.0,0.0


✅ **What to notice.** All three agree on the strong predictors (traffic, ads, income, competitors). Least squares
splits the "how big is the store" effect between `size` and `seats` in some arbitrary way and gives `tenure` a
non-zero coefficient it does not deserve; ridge shrinks every coefficient and shares the size effect across the
pair; LASSO shrinks too and sets some coefficients exactly to zero. Which of `size` and `seats` LASSO keeps is a
stand-in choice, not a finding. The penalty values here were set by hand; in the walkthrough and the PA they are
chosen by `GridSearchCV`.

---
## Where this goes next

1. **The sliders.** [*Shrink to Fit*](https://gato365.github.io/gsb5544_instructor_learn_prep/week_10/shrink_to_fit.html): the same coffee chain; drag the penalty and watch the coefficients,
   the training and CV error curves, and the coefficient paths; run the grid search fold by fold; open the test
   set once at the end.
2. **The code walkthrough** (*Penalized Regression: Ridge and LASSO*, the first part of the PA 10.2 notebook):
   the Ames houses with *all* 67 predictors, where plain least squares with polynomial terms falls apart under
   cross-validation; then `Ridge`, `Lasso` and `ElasticNet` in a pipeline, counting zero coefficients, and
   `GridSearchCV` over `alpha` (and `l1_ratio`). Its twelve **DISCUSS** prompts are the questions above in Ames
   clothing — Q4–Q9 of this notebook answer them.
3. **PA 10.2.** The loans data again, predicting `interest_rate` — this time with ridge, LASSO and elastic net on
   many predictors, at least one parameter tuned with `GridSearchCV`. Bring the ideas: standardize inside the
   pipeline, choose `alpha` by cross-validation, read a zero coefficient as "not needed given the others", and
   keep the test estimate separate from the tuning.

Course site: [https://gato365.github.io/gsb5544_instructor_learn_prep/](https://gato365.github.io/gsb5544_instructor_learn_prep/).